# Evaluate
Use this notebook to evaluate a keypoint detection model.
Set the following `keypoint_names` and `get_prediction` specific to your model and dataset.

In [4]:
import torch
import torch.nn as nn
import numpy as np
import cv2 as cv
from typing import Dict, Tuple
from pathlib import Path
from ultralytics import YOLO

In [5]:
keypoint_names = [
    'topLeft', 'bottomLeft', 'topRight', 'bottomRight',
]

### Ultralytics YOLO

In [ ]:
def get_predictions(model, image_path: str, normalize: bool = True) -> Dict:
    """
    Run YOLO inference and extract keypoints
    
    Args:
        model: YOLO model instance
        image_path: Path to image
        normalize: If True, normalize keypoints to [0, 1] range
    Returns:
        dict with:
            - keypoints_xy: np.array shape (4, 2)
            - confidence: np.array shape (4,)
    """
    
    results = model.predict(image_path, verbose=False)
    
    # Check if any detections
    if len(results[0].keypoints.xyn) == 0:
        # No detection - return zeros
        return {
            'keypoints_xy': np.zeros((4, 2)),
            'confidence': np.zeros((4,))
        }
    
    # Get first detection (assuming single box in image)
    pred_keypoints = results[0].keypoints.xyn[0].cpu().numpy() if normalize else results[0].keypoints.xy[0].cpu().numpy()
    
    # Get detection confidence if available
    confidence = results[0].keypoints.conf[0].cpu().numpy()
    
    return {
        'keypoints_xy': pred_keypoints,
        'confidence': confidence
    }

### Custom pytorch model

In [6]:
def get_predictions(model, image_path: str, normalize: bool = True) -> Dict:
    """
    Inference a custom pytorch model and extract keypoints
    
    Args:
        model: PyTorch model instance
        image_path: Path to image
        normalize: If True, normalize keypoints to [0, 1] range
    Returns:
        dict with:
            - keypoints_xy: np.array shape (4, 2)
    """

    if not normalize:
        raise NotImplementedError("Non-normalized keypoints not implemented.")
    
    img = cv.imread(image_path)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    img = cv.resize(img, (224, 224))
    img = img.astype('float32') / 255.0

    tensor_img = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)  # Shape: (1, 3, H, W)

    with torch.no_grad():
        preds = model(tensor_img)  # model output shape: (1, num_keypoints*2)
        
    preds = preds.detach().cpu().numpy().reshape(-1, 2)  # Shape: (num_keypoints, 2)

    return {
        'keypoints_xy': preds
    }

## Function definition

In [12]:
def parse_yolo_label(label_line: str) -> Dict:
    """
    Parse a single YOLO pose label line

    Args:
        label_line: String like "0 0.538 0.468 0.780 0.643 0.148 0.639 2 ..."

    Returns:
        dict with:
            - class_id: int
            - bbox: np.array shape (4,) [x_center, y_center, width, height]
            - keypoints_xy: np.array shape (4, 2) [x, y coordinates]
            - visibility: np.array shape (4,) [visibility flags]
    """
    values = label_line.strip().split()
    values = [float(v) for v in values]

    class_id = int(values[0])
    bbox = np.array(values[1:5])

    # Keypoints: groups of 3 (x, y, visibility)
    kpt_data = np.array(values[5:]).reshape(-1, 3)
    keypoints_xy = kpt_data[:, :2]  # Just x, y
    visibility = kpt_data[:, 2]      # Visibility flags

    return {
        'class_id': class_id,
        'bbox': bbox,
        'keypoints_xy': keypoints_xy,
        'visibility': visibility
    }


def parse_yolo_label_file(label_path: str) -> Dict:
    """Parse label from a .txt file"""
    with open(label_path, 'r') as f:
        line = f.read().strip()
    return parse_yolo_label(line)

def calculate_euclidean_distance(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    """
    Calculate Euclidean distance between predicted and ground truth keypoints

    Args:
        pred: shape (K, 2) - K keypoints, (x, y) coordinates
        gt: shape (K, 2)

    Returns:
        distances: shape (K,) - distance for each keypoint
    """
    diff = pred - gt
    distances = np.sqrt(np.sum(diff ** 2, axis=1))
    return distances

def calculate_single_metric(pred: np.ndarray, gt: np.ndarray, loss_fn: callable,
                           metric_name: str, visibility: np.ndarray) -> Dict:
    """
    Calculate a single mean and overall metric for a single image

    Args:
        pred: shape (K, 2)
        gt: shape (K, 2)
        loss_fn: function to compute the metric function output (shape (K, 2))
        metric_name: name of the metric
        visibility: shape (K,)

    Returns:
        dict with:
            - metric_name + '_overall': float
            - metric_name + '_per_keypoint': np.array shape (K,)
    """

    loss = loss_fn(pred, gt)
    visible_mask = (visibility == 2)

    if not np.any(visible_mask):
        return {
            f'{metric_name}_overall': np.nan,
            f'{metric_name}_per_keypoint': np.full(len(pred), np.nan)
        }

    per_kpt = np.mean(loss, axis=1) # loss xy averaged
    per_kpt = np.where(visible_mask, per_kpt, np.nan)
    overall = np.nanmean(per_kpt)

    return {
        f'{metric_name}_overall': float(overall),
        f'{metric_name}_per_keypoint': per_kpt
    }

def abs_loss(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    return np.abs(pred - gt)

def sq_loss(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    return (pred - gt) ** 2

def calculate_pck_single(pred: np.ndarray, gt: np.ndarray, visibility: np.ndarray,
                        threshold: float = 0.05) -> Dict:
    """
    Calculate PCK (Percentage of Correct Keypoints) for a single image

    Args:
        pred: shape (K, 2)
        gt: shape (K, 2)
        visibility: shape (K,)
        threshold: distance threshold (normalized, e.g., 0.05 = 5% of image)

    Returns:
        dict with:
            - pck: float (percentage of correct keypoints, 0-1)
            - correct_per_keypoint: np.array shape (K,) - bool array

    Interpretation:
        PCK@0.05 = 0.95 means 95% of visible keypoints within 5% of image size
    """
    pred = pred
    gt = gt
    distances = calculate_euclidean_distance(pred, gt)
    correct = distances < threshold
    correct = np.where(visibility == 2, correct, np.nan)
    pck = np.nanmean(correct)

    return {
        'pck': float(pck),
        'correct_per_keypoint': correct
    }

def evaluate_single_frame(pred_keypoints: np.ndarray,
                               gt_keypoints: np.ndarray,
                               visibility: np.ndarray = None,
                               thresholds: list = [0.02, 0.05, 0.10]) -> Dict:
    """
    Calculate ALL metrics for a single frame

    Args:
        pred_keypoints: shape (K, 2) - predicted keypoints
        gt_keypoints: shape (K, 2) - ground truth keypoints
        visibility: shape (K,) - visibility flags
        thresholds: list of PCK thresholds to evaluate

    Returns:
        dict with all metrics
    """
    metrics = {}

    # MSE
    mse_result = calculate_single_metric(pred_keypoints, gt_keypoints, sq_loss, 'mse', visibility)
    metrics['mse_overall'] = mse_result['mse_overall']
    metrics['mse_per_keypoint'] = mse_result['mse_per_keypoint']

    # MAE
    mae_result = calculate_single_metric(pred_keypoints, gt_keypoints, abs_loss, 'mae', visibility)
    metrics['mae_overall'] = mae_result['mae_overall']
    metrics['mae_per_keypoint'] = mae_result['mae_per_keypoint']

    # PCK at multiple thresholds
    for thresh in thresholds:
        pck_result = calculate_pck_single(pred_keypoints, gt_keypoints, visibility, thresh)
        metrics[f'pck_{thresh}'] = pck_result['pck']
        metrics[f'pck_{thresh}_per_keypoint'] = pck_result['correct_per_keypoint']

    # Per-keypoint Euclidean distances (useful for analysis)
    metrics['distances'] = calculate_euclidean_distance(pred_keypoints, gt_keypoints)

    return metrics

## Single image evaluation example

### Instantiate YOLO

In [ ]:
# 1. Load model
model_path = 'last.pt'
model = YOLO(model_path)
print(f"\nLoaded model: {model_path}")

### Instantiate model

In [7]:
model_path = 'model_with_norm_traced.pt'

try:
    model = torch.jit.load(model_path, map_location='cpu')
    model.eval()
    print(f"\nLoaded model: {model_path}")
except Exception as e:
    print(f"Error loading model: {e}")


Loaded model: model_with_norm_traced.pt


### Evaluate

In [8]:
# 2. Load image and label
image_path = 'ARVP_Front_front_2025-08-12_15-39-45_UTC_394053732.jpeg'
label_path = 'ARVP_Front_front_2025-08-12_15-39-45_UTC_394053732.txt'

In [9]:
print(f"Image: {image_path}")
print(f"Label: {label_path}")

# 3. Parse ground truth
gt_data = parse_yolo_label_file(label_path)

print("\nGround Truth Keypoints:")
print(gt_data)
gt_keypoints = gt_data['keypoints_xy']
visibility = gt_data['visibility']

# 4. Get predictions
pred_data = get_predictions(model, image_path, normalize=True)

print(f"pred_data: {pred_data}")
pred_keypoints = pred_data['keypoints_xy']

# 5. Calculate metrics
metrics = evaluate_single_frame(
    pred_keypoints, 
    gt_keypoints, 
    visibility,
    thresholds=[0.02, 0.05, 0.10]
)

# 6. Display results
print(f"\nOverall Metrics:")
print(f"  MSE: {metrics['mse_overall']:.6f}")
print(f"  MAE: {metrics['mae_overall']:.6f}")
print(f"  PCK@0.02: {metrics['pck_0.02']:.4f}")
print(f"  PCK@0.05: {metrics['pck_0.05']:.4f}")
print(f"  PCK@0.10: {metrics['pck_0.1']:.4f}")

print(f"\nPer-Keypoint Analysis:")
print(f"  {'Keypoint':<12} {'Distance':<12} {'MSE':<12} {'Correct@0.05'}")
print(f"  {'-'*50}")


for i, name in enumerate(keypoint_names):
    if visibility[i] != 2:
        continue  # Skip non-visible keypoints
    dist = metrics['distances'][i]
    mse = metrics['mse_per_keypoint'][i]
    correct = 'Yes' if metrics['pck_0.05_per_keypoint'][i] else 'No'
    print(f"  {name:<12} {dist:<12.6f} {mse:<12.6f} {correct}")


Image: ARVP_Front_front_2025-08-12_15-39-45_UTC_394053732.jpeg
Label: ARVP_Front_front_2025-08-12_15-39-45_UTC_394053732.txt


NameError: name 'parse_yolo_label_file' is not defined

## Evaluate on Test set

In [10]:
from ultralytics import YOLO
import json
from datetime import datetime

def evaluate_dataset(model, test_images_dir: str, test_labels_dir: str,
                    save_results: bool = True, output_path: str = None) -> Dict:
    """
    Evaluate model on entire test dataset
    
    Args:
        model: 
        test_images_dir: Directory with test images
        test_labels_dir: Directory with test labels (.txt files)
        save_results: Whether to save results to JSON
        output_path: Where to save results (optional)
        benchmark_resources: Whether to run resource usage benchmark
        num_benchmark_iterations: Number of iterations for resource benchmark
    Returns:
        dict with aggregated metrics across all images
    """
    
    test_images_dir = Path(test_images_dir)
    test_labels_dir = Path(test_labels_dir)
    
    test_images = []
    for img_file in test_images_dir.glob('*'):
        test_images.append(img_file)
    print(f"Found {len(test_images)} test images.")
    
    images_with_labels = []
    for img_path in test_images:
        label_path = test_labels_dir / (img_path.stem + '.txt')
        if label_path.exists():
            images_with_labels.append((img_path, label_path))
        else:
            print(f"Warning: No label file for image {img_path.name}, skipping.")
            
    print(f"Evaluating {len(images_with_labels)} images with labels.")
    
    test_images = images_with_labels
    all_metrics = []
    all_distances = []
    all_mse_per_kpt = []
    all_pck_per_kpt = {
        'pck_0.02': [],
        'pck_0.05': [],
        'pck_0.1': []
    }
    
    failed_images = []
    
    for idx, (img_path, label_path) in enumerate(test_images):
        if (idx + 1) % 25 == 0:
            print(f"  Progress: {idx + 1}/{len(test_images)}")
                
        try:
            gt_data = parse_yolo_label_file(str(label_path))
            
            pred_data = get_predictions(model, str(img_path), normalize=True)
            
            metrics = evaluate_single_frame(
                pred_data['keypoints_xy'],
                gt_data['keypoints_xy'],
                gt_data['visibility'],
                thresholds=[0.02, 0.05, 0.10]
            )
            
            all_metrics.append({
                'image': img_path.name,
                'mse': metrics['mse_overall'],
                'mae': metrics['mae_overall'],
                'pck_0.02': metrics.get('pck_0.02', 0.0),
                'pck_0.05': metrics.get('pck_0.05', 0.0),
                'pck_0.1': metrics.get('pck_0.1', 0.0),
            })
            all_distances.append(metrics['distances'])
            all_mse_per_kpt.append(metrics['mse_per_keypoint'])
            
            all_pck_per_kpt['pck_0.02'].append(metrics['pck_0.02_per_keypoint'])
            all_pck_per_kpt['pck_0.05'].append(metrics['pck_0.05_per_keypoint'])
            all_pck_per_kpt['pck_0.1'].append(metrics['pck_0.1_per_keypoint'])
        
        except Exception as e:
            print(f"  WARNING: Failed on {img_path.name}: {e}")
            failed_images.append(str(img_path.name))
            
    print(f"Evaluated {len(all_metrics)} images.")
    
    if failed_images:
        print(f"Failed on {len(failed_images)} images: {failed_images}")
            
    # Aggregate overall metrics
    all_distances = np.array(all_distances)
    all_mse_per_kpt = np.array(all_mse_per_kpt)
    
    overall_metrics = {
        'mse_overall': float(np.mean([m['mse'] for m in all_metrics])),
        'mae_overall': float(np.mean([m['mae'] for m in all_metrics])),
        'pck_0.02': float(np.nanmean([m['pck_0.02'] for m in all_metrics])),
        'pck_0.05': float(np.nanmean([m['pck_0.05'] for m in all_metrics])),
        'pck_0.1': float(np.nanmean([m['pck_0.1'] for m in all_metrics])),
        'avg_distances_per_keypoint': np.nanmean(all_distances, axis=0).tolist(),
        'avg_mse_per_keypoint': np.nanmean(all_mse_per_kpt, axis=0).tolist(),
        'avg_pck_per_keypoint': {
            'pck_0.02': np.nanmean(all_pck_per_kpt['pck_0.02'], axis=0).tolist(),
            'pck_0.05': np.nanmean(all_pck_per_kpt['pck_0.05'], axis=0).tolist(),
            'pck_0.1': np.nanmean(all_pck_per_kpt['pck_0.1'], axis=0).tolist(),
        },
        'num_images': len(all_metrics),
        'num_failed': len(failed_images),
        'model_path': str(model_path),
        'test_images_dir': str(test_images_dir),
        'test_labels_dir': str(test_labels_dir),
        'timestamp': datetime.now().isoformat(),
    }
    
    print("\nOverall Dataset Metrics:")
    print(f"Total Images Evaluated: {overall_metrics['num_images']}")
    print(f"Failed Images: {overall_metrics['num_failed']}")
    print(f"  MSE: {overall_metrics['mse_overall']:.6f}")
    print(f"  MAE: {overall_metrics['mae_overall']:.6f}")
    print(f"  PCK@0.02: {overall_metrics['pck_0.02']:.4f}")
    print(f"  PCK@0.05: {overall_metrics['pck_0.05']:.4f}")
    print(f"  PCK@0.10: {overall_metrics['pck_0.1']:.4f}")
    
    print("\nAverage Per-Keypoint Metrics:")
    print(f"  {'Keypoint':<12} {'Avg Distance':<15} {'Avg MSE':<15} {'PCK@0.05'}")
    print(f"  {'-'*60}")
    for i, name in enumerate(keypoint_names):
        avg_dist = overall_metrics['avg_distances_per_keypoint'][i]
        avg_mse = overall_metrics['avg_mse_per_keypoint'][i]
        pck_05 = overall_metrics['avg_pck_per_keypoint']['pck_0.05'][i]
        print(f"  {name:<12} {avg_dist:<15.6f} {avg_mse:<15.6f} {pck_05:.4f}")
        
 
    # Save results to JSON
    if save_results:
        if output_path is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_path = f"evaluation_results_{timestamp}.json"
        
        with open(output_path, 'w') as f:
            json.dump(overall_metrics, f, indent=4)
        
        print(f"\nSaved evaluation results to: {output_path}")
        
    return overall_metrics

In [13]:
json = evaluate_dataset(
    model, 
    test_images_dir='val/images',
    test_labels_dir='val/labels',
    save_results=False
)

Found 149 test images.
Evaluating 146 images with labels.
  Progress: 25/146
  Progress: 50/146
  Progress: 75/146
  Progress: 100/146
  Progress: 125/146
Evaluated 146 images.

Overall Dataset Metrics:
Total Images Evaluated: 146
Failed Images: 0
  MSE: 0.001910
  MAE: 0.029790
  PCK@0.02: 0.2260
  PCK@0.05: 0.6952
  PCK@0.10: 0.9201

Average Per-Keypoint Metrics:
  Keypoint     Avg Distance    Avg MSE         PCK@0.05
  ------------------------------------------------------------
  topLeft      0.158048        0.001283        0.7303
  bottomLeft   0.272449        0.002474        0.6212
  topRight     0.139366        0.001533        0.7379
  bottomRight  0.299273        0.002934        0.4255
